# Open and Test the Database Connection

## Prerequisites

Ensure the following are in place before proceeding:

- You have SSH access to the server where BioSoundsLab is running.
- SSH client is installed on your local machine
- SSH access to the server is configured, including an entry in your `~/.ssh/config` file for the server connection
- The `docker-compose.production.yaml` file exists on the server and contains the database service configuration
- The BioSoundsLab instance is running in Docker, and its environment file (containing `DB_USERNAME` and `DB_PASSWORD`) is accessible
- Python environment is set up on your local machine with necessary packages (e.g., `python-dotenv`, database drivers)

---

## Steps

### 1. Export the Database Port in Docker

The database port is typically not exposed outside the Docker network. To make it accessible from the host server, edit the corresponding service configuration in the `docker-compose.production.yaml` file on the server. Add or modify the `ports` section to expose the database port:
```
 yaml
  services:
   db:
    ports:
      - "PORT\:PORT"
```

Replace PORT with the actual database port (e.g., 5432). Save the file and restart the Docker containers or force recreate the service.

### 2. Set Up an SSH Tunnel
Run on your local machine:

```ssh -f -N -L PORT:localhost:PORT SERVER-CONNECTION-NAME```

* SERVER-CONNECTION-NAME: SSH config name for the server
* PORT: The database port

If issues arise, specify the SSH config path:
```
ssh -F PATH-TO/config -f -N -L PORT:localhost:PORT USERNAME@SERVER-CONNECTION-NAME
```

* PATH-TO/config: Path to your SSH config file (e.g., ~/.ssh/config)
* USERNAME: Remote username

### 3. Configure the Local Environment
On your local machine, create a .env file in the notebooks directory with:
```
DB_USERNAME=your_database_username
DB_PASSWORD=your_database_password
```

### 4. Set Database Configuration Variables
In your code or DB_CONFIG:
* port: Local port from the SSH tunnel (same as PORT in the SSH command)
* instance_name: Name of the BioSoundsLab instance (used in filenames for stored data)


### Open the connection

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Load the database username and password
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Read the username and password
username = os.getenv("DB_USERNAME")
password = os.getenv("DB_PASSWORD")

DB_CONFIG = {
    'host': 'localhost',
    'port': 8432,  # the port opened using the ssh tunnel
    'user': username,
    'password': password,
    'database': 'ecomon'
}

# make sure this is the name of the instance you want to query, 
# it will be used in the name of the output file.
instance_name = "MGB"  
model_name = 'BNP_v3_0_glob11k_prev3_cw5s_sz2s'

## Test the connection

In [ ]:
from bslquery import BioSoundsLabDB

# Initialize the database helper
db = BioSoundsLabDB(DB_CONFIG, verbose=True)

# Test the connection
db.test_connection()

Once done with querying the database, __close the port and restart the db service__.